# Generate Fixed Attack/Anomaly Datasets

Applies the **same fixed pipeline** used in `ndn_pipeline.ipynb` (delta-based `cache_hit_ratio`, `satisfaction_ratio`, `unsatisfied_ratio` + per-subset clip bounds) to the attack scenarios.

Outputs (saved alongside `normal_traffic_features.csv` in `DATASET_DIR`):
- `anomaly_traffic_features.csv` — IFA + CP combined
- `ifa_attack_features.csv` — IFA only
- `cp_attack_features.csv` — CP only
- `ndn_mixed_normal_anomaly_features.csv` — normal + all attacks combined & shuffled

In [1]:
# ── Imports ───────────────────────────────────────────────────────────────────
import glob
import json
import logging
import warnings
from pathlib import Path
from typing import List, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s  %(message)s')
logger = logging.getLogger(__name__)
print('✅ Imports OK')

✅ Imports OK


In [2]:
# ── Paths — adjust ROOT_DIR to match your layout ──────────────────────────────
ROOT_DIR    = Path('/Users/ankitpokhrel/Desktop/NDN-Toolkit')  
LOG_ROOT    = ROOT_DIR / 'Logs'
DATASET_DIR = ROOT_DIR / 'Datasets'
DATASET_DIR.mkdir(parents=True, exist_ok=True)

# ── Feature / rate column lists (must match ndn_pipeline.ipynb exactly) ───────
FEATURE_COLS = [
    'pit_size', 'pit_growth_rate', 'cs_size',
    'cache_hit_ratio', 'satisfaction_ratio', 'unsatisfied_ratio',
    'in_interests_rate', 'out_interests_rate', 'in_data_rate', 'nack_rate',
]
RATE_COLS = [
    'pit_growth_rate', 'in_interests_rate', 'out_interests_rate',
    'in_data_rate', 'nack_rate',
]
REQUIRED_RAW_COLS = [
    'timestamp', 'node',
    'nPitEntries', 'nInInterests', 'nOutInterests',
    'nInData', 'nInNacks', 'nOutNacks',
    'nSatisfiedInterests', 'nUnsatisfiedInterests',
    'nCsEntries', 'nHits', 'nMisses',
]

WARMUP_SECONDS   = 600      # 10-min warmup for normal; set to 0 for attack
CLIP_QUANTILE_LO = 0.01
CLIP_QUANTILE_HI = 0.99

# Scenario groups
NORMAL_SCENARIOS = ['logs1', 'logs_mesh1', 'logs_dumbbell1']
IFA_SCENARIOS    = ['logs_dumbbell_ifa']
CP_SCENARIOS     = ['logs_tree_cp']
ATTACK_SCENARIOS = IFA_SCENARIOS + CP_SCENARIOS

print(f'📁 Log root  : {LOG_ROOT}')
print(f'💾 Datasets  : {DATASET_DIR}')

📁 Log root  : /Users/ankitpokhrel/Desktop/NDN-Toolkit/Logs
💾 Datasets  : /Users/ankitpokhrel/Desktop/NDN-Toolkit/Datasets


In [3]:
# ── Helper functions (copied verbatim from ndn_pipeline.ipynb) ────────────────

def parse_jsonl_file(filepath: Path) -> pd.DataFrame:
    records = []
    with open(filepath) as fh:
        for lineno, line in enumerate(fh, 1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                logger.warning(f'  Bad JSON at {filepath.name}:{lineno} — skipped')
    return pd.DataFrame(records) if records else pd.DataFrame()


def load_raw_logs(scenarios: List[str], log_root: Path = LOG_ROOT) -> pd.DataFrame:
    metric_subdirs = ['normal_traffic_metrics', 'cache_pollution_metrics']
    found = []
    for scenario in scenarios:
        base = log_root / scenario
        for sub in metric_subdirs:
            pattern = str(base / '**' / sub / '*_metrics.jsonl')
            found.extend(glob.glob(pattern, recursive=True))

    if not found:
        raise FileNotFoundError(
            f'No *_metrics.jsonl files found under {log_root} for scenarios: {scenarios}'
        )
    logger.info(f'Found {len(found)} JSONL files across {len(scenarios)} scenario(s)')

    frames = []
    for fp in tqdm(sorted(set(found)), desc='Parsing logs'):
        p  = Path(fp)
        df = parse_jsonl_file(p)
        if df.empty:
            continue
        node_from_path = p.parent.parent.name
        if 'node' not in df.columns or df['node'].isna().all():
            df['node'] = node_from_path
        df['_scenario']    = p.parts[-4]
        df['_source_file'] = fp
        frames.append(df)

    if not frames:
        raise ValueError('All log files were empty or unreadable.')

    combined = pd.concat(frames, ignore_index=True)
    logger.info(f'Raw records loaded: {len(combined):,}')
    return combined


def clean_logs(df: pd.DataFrame, warmup_seconds: int = WARMUP_SECONDS) -> pd.DataFrame:
    n_raw = len(df)

    if 'error' in df.columns:
        mask = df['error'].notna()
        logger.info(f'  Error rows dropped   : {mask.sum():,}')
        df = df[~mask].drop(columns=['error'])

    if 'node' not in df.columns:
        raise ValueError("'node' column is missing.")
    df = df.dropna(subset=['node'])

    available = [c for c in REQUIRED_RAW_COLS if c in df.columns]
    missing   = [c for c in REQUIRED_RAW_COLS if c not in df.columns]
    if missing:
        logger.warning(f'  Missing columns: {missing}')

    df = df.dropna(subset=[c for c in available if c not in ('timestamp', 'node')])
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df = df.dropna(subset=['timestamp'])

    metric_cols = [c for c in available if c not in ('timestamp', 'node')]
    for col in metric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=metric_cols)

    df = df.sort_values(['node', 'timestamp']).reset_index(drop=True)
    before_dedup = len(df)
    df = df.drop_duplicates(subset=['node', 'timestamp'])

    # Warmup removal
    if warmup_seconds > 0:
        df['_elapsed'] = df.groupby('node')['timestamp'].transform(
            lambda t: (t - t.min()).dt.total_seconds()
        )
        df = df[df['_elapsed'] >= warmup_seconds].drop(columns=['_elapsed'])

    logger.info(f'  Raw: {n_raw:,}  →  Clean: {len(df):,}  '
                f'(dupes removed: {before_dedup - len(df):,})')
    return df.reset_index(drop=True)


def engineer_features_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Fixed feature engineering:
    - cache_hit_ratio, satisfaction_ratio, unsatisfied_ratio are DELTA-based
      (matches ndn_pipeline.ipynb's engineer_features_df, not the old
       instantaneous version in data_preprocessing.ipynb).
    """
    feat_frames = []

    for node, grp in tqdm(df.groupby('node'), desc='Feature engineering'):
        grp = grp.sort_values('timestamp').copy()
        dt  = grp['timestamp'].diff().dt.total_seconds()

        # Absolute
        grp['pit_size'] = grp['nPitEntries']
        grp['cs_size']  = grp['nCsEntries']

        # Rate features
        grp['pit_growth_rate']    = grp['nPitEntries'].diff()  / dt
        grp['in_interests_rate']  = grp['nInInterests'].diff() / dt
        grp['out_interests_rate'] = grp['nOutInterests'].diff()/ dt
        grp['in_data_rate']       = grp['nInData'].diff()      / dt
        grp['nack_rate']          = (grp['nInNacks'] + grp['nOutNacks']).diff() / dt

        # Delta-based ratio features (THE FIX vs data_preprocessing.ipynb)
        d_hits   = grp['nHits'].diff()
        d_misses = grp['nMisses'].diff()
        d_cache  = d_hits + d_misses
        grp['cache_hit_ratio'] = np.where(d_cache > 0, d_hits / d_cache, 0.0)

        d_sat   = grp['nSatisfiedInterests'].diff()
        d_unsat = grp['nUnsatisfiedInterests'].diff()
        d_total = d_sat + d_unsat
        grp['satisfaction_ratio'] = np.where(d_total > 0, d_sat   / d_total, 0.0)
        grp['unsatisfied_ratio']  = np.where(d_total > 0, d_unsat / d_total, 0.0)

        feat_frames.append(grp)

    feat_df = pd.concat(feat_frames, ignore_index=True)

    provenance = [c for c in ['_scenario', 'source_scenarios', 'elapsed_sec']
                  if c in feat_df.columns]
    keep = ['timestamp', 'node'] + FEATURE_COLS + provenance
    feat_df = feat_df[[c for c in keep if c in feat_df.columns]].copy()

    n_before = len(feat_df)
    feat_df  = feat_df.dropna(subset=FEATURE_COLS).reset_index(drop=True)
    logger.info(f'  Feature rows after diff-drop: {len(feat_df):,}  '
                f'(dropped {n_before - len(feat_df):,})')

    # Clip rate columns and store bounds as DataFrame attribute
    clip_bounds = {}
    for col in RATE_COLS:
        lo = float(feat_df[col].quantile(CLIP_QUANTILE_LO))
        hi = float(feat_df[col].quantile(CLIP_QUANTILE_HI))
        feat_df[col] = feat_df[col].clip(lo, hi)
        clip_bounds[col] = {'lo': lo, 'hi': hi}

    feat_df.attrs['clip_bounds'] = clip_bounds
    return feat_df


def prepare_dataset(
    scenarios: List[str],
    label: str = 'normal',
    log_root: Path = LOG_ROOT,
    warmup_seconds: int = WARMUP_SECONDS,
    save_path: Optional[Path] = None,
) -> pd.DataFrame:
    logger.info(f"\n{'='*60}")
    logger.info(f'Preparing dataset: {label}')
    logger.info(f'Scenarios: {scenarios}')

    raw   = load_raw_logs(scenarios, log_root)
    clean = clean_logs(raw, warmup_seconds)
    feats = engineer_features_df(clean)

    feats['source_scenarios'] = label
    logger.info(f'Final shape: {feats.shape}')
    logger.info(f"Nodes: {sorted(feats['node'].unique())}")

    if save_path:
        feats.drop(columns=['_scenario'], errors='ignore').to_csv(save_path, index=False)
        logger.info(f'Saved → {save_path}')

    return feats


print('✅ All functions defined')

✅ All functions defined


In [4]:
# ── 1. IFA attack dataset ─────────────────────────────────────────────────────
# warmup_seconds=0 because attacks may start immediately (mirrors data_preprocessing.ipynb)
ifa_df = prepare_dataset(
    scenarios      = IFA_SCENARIOS,
    label          = 'ifa_attack',
    warmup_seconds = 0,
    save_path      = DATASET_DIR / 'ifa_attack_features.csv',
)

print(f'\n✅ IFA rows : {len(ifa_df):,}')
ifa_df.head()

INFO  
INFO  Preparing dataset: ifa_attack
INFO  Scenarios: ['logs_dumbbell_ifa']
INFO  Found 10 JSONL files across 1 scenario(s)
Parsing logs: 100%|██████████| 10/10 [00:00<00:00, 100.26it/s]
INFO  Raw records loaded: 20,543
INFO    Raw: 20,543  →  Clean: 20,543  (dupes removed: 0)
Feature engineering: 100%|██████████| 10/10 [00:00<00:00, 617.71it/s]
INFO    Feature rows after diff-drop: 20,533  (dropped 10)
INFO  Final shape: (20533, 14)
INFO  Nodes: ['bottleneck', 'c1', 'c2', 'c3', 'p1', 'p2', 'r1', 'r2', 'r3', 'r4']
INFO  Saved → /Users/ankitpokhrel/Desktop/NDN-Toolkit/Datasets/ifa_attack_features.csv



✅ IFA rows : 20,533


,timestamp,node,pit_size,pit_growth_rate,cs_size,cache_hit_ratio,satisfaction_ratio,unsatisfied_ratio,in_interests_rate,out_interests_rate,in_data_rate,nack_rate,_scenario,source_scenarios
0,2026-03-13 13:44:15.396519,bottleneck,3,0.0,152,0.000000,1.0,0.0,4.922701,2.953621,2.953621,0.0,logs_dumbbell_ifa,ifa_attack
1,2026-03-13 13:44:17.411146,bottleneck,3,0.0,165,0.034483,1.0,0.0,7.899890,5.956220,4.946726,0.0,logs_dumbbell_ifa,ifa_attack
2,2026-03-13 13:44:19.424967,bottleneck,3,0.0,165,0.000000,1.0,0.0,4.965685,2.979411,2.979411,0.0,logs_dumbbell_ifa,ifa_attack
3,2026-03-13 13:44:21.447898,bottleneck,3,0.0,165,0.000000,1.0,0.0,4.943322,2.965993,3.460326,0.0,logs_dumbbell_ifa,ifa_attack
4,2026-03-13 13:44:23.462059,bottleneck,3,0.0,168,0.000000,1.0,0.0,6.454300,4.468362,3.971877,0.0,logs_dumbbell_ifa,ifa_attack


In [5]:
# ── 2. CP attack dataset ──────────────────────────────────────────────────────
cp_df = prepare_dataset(
    scenarios      = CP_SCENARIOS,
    label          = 'cp_attack',
    warmup_seconds = 0,
    save_path      = DATASET_DIR / 'cp_attack_features.csv',
)

print(f'\n✅ CP rows  : {len(cp_df):,}')
cp_df.head()

INFO  
INFO  Preparing dataset: cp_attack
INFO  Scenarios: ['logs_tree_cp']
INFO  Found 12 JSONL files across 1 scenario(s)
Parsing logs: 100%|██████████| 12/12 [00:00<00:00, 170.01it/s]
INFO  Raw records loaded: 16,228
INFO    Raw: 16,228  →  Clean: 16,228  (dupes removed: 0)
Feature engineering: 100%|██████████| 12/12 [00:00<00:00, 656.75it/s]
INFO    Feature rows after diff-drop: 16,216  (dropped 12)
INFO  Final shape: (16216, 14)
INFO  Nodes: ['c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'p1', 'p2', 'r1', 'r2', 'r3', 'r4']
INFO  Saved → /Users/ankitpokhrel/Desktop/NDN-Toolkit/Datasets/cp_attack_features.csv



✅ CP rows  : 16,216


,timestamp,node,pit_size,pit_growth_rate,cs_size,cache_hit_ratio,satisfaction_ratio,unsatisfied_ratio,in_interests_rate,out_interests_rate,in_data_rate,nack_rate,_scenario,source_scenarios
0,2026-03-13 15:25:43.683473,c1,2,0.0,103,0.0,1.0,0.0,8.404798,6.903220,5.442712,0.0,logs_tree_cp,cp_attack
1,2026-03-13 15:25:45.715139,c1,3,0.0,103,0.0,1.0,0.0,5.906483,3.937655,3.445448,0.0,logs_tree_cp,cp_attack
2,2026-03-13 15:25:47.742703,c1,3,0.0,103,0.0,1.0,0.0,4.932027,2.959216,2.466013,0.0,logs_tree_cp,cp_attack
3,2026-03-13 15:25:49.775643,c1,3,0.0,107,0.0,1.0,0.0,7.378476,5.410883,4.918984,0.0,logs_tree_cp,cp_attack
4,2026-03-13 15:25:51.797646,c1,3,0.0,113,0.0,1.0,0.0,8.404798,6.903220,5.442712,0.0,logs_tree_cp,cp_attack


In [6]:
# ── 3. Combined anomaly dataset (IFA + CP) ────────────────────────────────────
anomaly_df = pd.concat([ifa_df, cp_df], ignore_index=True)
anomaly_out = DATASET_DIR / 'anomaly_traffic_features.csv'
anomaly_df.drop(columns=['_scenario'], errors='ignore').to_csv(anomaly_out, index=False)

print(f'✅ Anomaly (IFA+CP) rows : {len(anomaly_df):,}')
print(f'   Saved → {anomaly_out}')
anomaly_df.head()

✅ Anomaly (IFA+CP) rows : 36,749
   Saved → /Users/ankitpokhrel/Desktop/NDN-Toolkit/Datasets/anomaly_traffic_features.csv


,timestamp,node,pit_size,pit_growth_rate,cs_size,cache_hit_ratio,satisfaction_ratio,unsatisfied_ratio,in_interests_rate,out_interests_rate,in_data_rate,nack_rate,_scenario,source_scenarios
0,2026-03-13 13:44:15.396519,bottleneck,3,0.0,152,0.000000,1.0,0.0,4.922701,2.953621,2.953621,0.0,logs_dumbbell_ifa,ifa_attack
1,2026-03-13 13:44:17.411146,bottleneck,3,0.0,165,0.034483,1.0,0.0,7.899890,5.956220,4.946726,0.0,logs_dumbbell_ifa,ifa_attack
2,2026-03-13 13:44:19.424967,bottleneck,3,0.0,165,0.000000,1.0,0.0,4.965685,2.979411,2.979411,0.0,logs_dumbbell_ifa,ifa_attack
3,2026-03-13 13:44:21.447898,bottleneck,3,0.0,165,0.000000,1.0,0.0,4.943322,2.965993,3.460326,0.0,logs_dumbbell_ifa,ifa_attack
4,2026-03-13 13:44:23.462059,bottleneck,3,0.0,168,0.000000,1.0,0.0,6.454300,4.468362,3.971877,0.0,logs_dumbbell_ifa,ifa_attack


In [7]:
# ── 4. Mixed dataset (normal + all attacks, shuffled) ─────────────────────────
# Load the already-generated normal fixed CSV if it exists; otherwise rebuild it.
normal_csv = DATASET_DIR / 'normal_traffic_features.csv'

if normal_csv.exists():
    normal_df = pd.read_csv(normal_csv)
    print(f'📂 Loaded existing normal dataset: {len(normal_df):,} rows')
else:
    print('ℹ️  normal_traffic_features.csv not found — rebuilding from logs …')
    normal_df = prepare_dataset(
        scenarios      = NORMAL_SCENARIOS,
        label          = 'normal',
        warmup_seconds = WARMUP_SECONDS,
        save_path      = normal_csv,
    )

mixed_df = pd.concat([normal_df, anomaly_df], ignore_index=True)
mixed_df = mixed_df.sample(frac=1.0, random_state=42).reset_index(drop=True)

mixed_out = DATASET_DIR / 'ndn_mixed_normal_anomaly_features.csv'
mixed_df.drop(columns=['_scenario'], errors='ignore').to_csv(mixed_out, index=False)

print(f'\n✅ Mixed dataset rows: {len(mixed_df):,}')
print(f'   Normal      : {len(normal_df):,}')
print(f'   IFA attack  : {len(ifa_df):,}')
print(f'   CP attack   : {len(cp_df):,}')
print(f'   Saved → {mixed_out}')
mixed_df.head()

INFO  
INFO  Preparing dataset: normal
INFO  Scenarios: ['logs1', 'logs_mesh1', 'logs_dumbbell1']
INFO  Found 34 JSONL files across 3 scenario(s)


ℹ️  normal_traffic_features.csv not found — rebuilding from logs …


Parsing logs: 100%|██████████| 34/34 [00:00<00:00, 61.00it/s]
INFO  Raw records loaded: 130,610
INFO    Error rows dropped   : 45
INFO    Raw: 130,610  →  Clean: 126,436  (dupes removed: 4,129)
Feature engineering: 100%|██████████| 13/13 [00:00<00:00, 514.14it/s]
INFO    Feature rows after diff-drop: 126,423  (dropped 13)
INFO  Final shape: (126423, 14)
INFO  Nodes: ['bottleneck', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'p1', 'p2', 'r1', 'r2', 'r3', 'r4']
INFO  Saved → /Users/ankitpokhrel/Desktop/NDN-Toolkit/Datasets/normal_traffic_features.csv



✅ Mixed dataset rows: 163,172
   Normal      : 126,423
   IFA attack  : 20,533
   CP attack   : 16,216
   Saved → /Users/ankitpokhrel/Desktop/NDN-Toolkit/Datasets/ndn_mixed_normal_anomaly_features.csv


,timestamp,node,pit_size,pit_growth_rate,cs_size,cache_hit_ratio,satisfaction_ratio,unsatisfied_ratio,in_interests_rate,out_interests_rate,in_data_rate,nack_rate,_scenario,source_scenarios
0,2026-03-13 06:37:32.164291,p1,3.0,0.0,306.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,logs_dumbbell1,normal
1,2026-03-13 06:43:45.241317,r2,3.0,0.0,428.0,0.0,1.0,0.0,9.841667,6.397084,6.397084,0.0,logs_dumbbell1,normal
2,2026-03-13 06:08:03.714990,bottleneck,3.0,0.0,247.0,0.0,1.0,0.0,17.756817,10.654090,9.766249,0.0,logs_dumbbell1,normal
3,2026-03-13 04:08:11.425155,c5,3.0,0.0,331.0,0.0,1.0,0.0,4.944192,2.966515,2.966515,0.0,logs_mesh1,normal
4,2026-03-13 05:15:54.671748,r1,3.0,0.0,1694.0,0.0,1.0,0.0,4.931798,2.959079,2.465899,0.0,logs_mesh1,normal


In [8]:
# ── 5. Summary ────────────────────────────────────────────────────────────────
print('\n' + '='*60)
print('  Generated files (all in Datasets/):')
print('='*60)
outputs = [
    'ifa_attack_features.csv',
    'cp_attack_features.csv',
    'anomaly_traffic_features.csv',
    'ndn_mixed_normal_anomaly_features.csv',
]
for fname in outputs:
    p = DATASET_DIR / fname
    exists = '✅' if p.exists() else '❌ MISSING'
    size   = f'{p.stat().st_size / 1024:.1f} KB' if p.exists() else ''
    print(f'  {exists}  {fname}  {size}')

print('\nFeature columns used:')
for col in FEATURE_COLS:
    print(f'  • {col}')


  Generated files (all in Datasets/):
  ✅  ifa_attack_features.csv  2465.1 KB
  ✅  cp_attack_features.csv  1918.6 KB
  ✅  anomaly_traffic_features.csv  4383.6 KB
  ✅  ndn_mixed_normal_anomaly_features.csv  18440.3 KB

Feature columns used:
  • pit_size
  • pit_growth_rate
  • cs_size
  • cache_hit_ratio
  • satisfaction_ratio
  • unsatisfied_ratio
  • in_interests_rate
  • out_interests_rate
  • in_data_rate
  • nack_rate
